# Sparse selection and stability

This notebook runs the Phase 3 sparse-selection and stability stage. During automated validation it resolves placeholder paths to the deterministic demo dataset; once local real-data paths are configured it runs against the real case-study artifacts.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from bsm_rfm import build_manuscript_notebook_context, run_sparse_selection_stability_stage

context = build_manuscript_notebook_context(REPO_ROOT, "06_sparse_selection_and_stability.ipynb")
result = run_sparse_selection_stability_stage(context)
result.sparse_selection.summary

### Step 1: Run De-Biased LASSO with Stability Analysis

Select stable features via repeated LASSO on bootstrap samples. Use de-biased LASSO to correct for selection bias.

**Process:**
1. For each of B bootstrap samples (~100):
   a. Fit LASSO on bootstrap resample
   b. Record which features are selected
   c. Estimate coefficients using de-biased correction
2. Compute feature selection frequency (e.g., selected in 50+ of 100 runs)
3. Keep only stable features (freq >= threshold)

**Parameters:**
- `lasso_repetitions`: Number of bootstrap samples (default 100)
- `selection_frequency_threshold`: Min frequency to keep (default 0.5, i.e., 50%)
- `lambda_rule`: LASSO regularization strength. 'cv1se' (default), 'cvmin', or manual value

**Output:**
- `stability_summary`: Selection frequency for each feature
- `selected_features_stable`: Final stable feature set (~50-100 features)

In [ ]:
result.sparse_selection.final_stable_support.head()

### Step 2: Apply HC3 Heteroscedasticity Filter

Filter stable features via robust variance criterion to remove inflated-variance features.

**HC3 filter:**
1. Fit model on stable features
2. Compute HC3 robust variance estimates for each coefficient
3. Keep features with HC3 score <= threshold (lower variance = better)
4. Typically removes 10-20% of stable features

**Output:**
- Final selected features: stable + HC3-filtered (~50-100 features)
- Handoff CSV ready for OLS